In [1]:
#Build a tiny evaluation dataset

import pandas as pd

eval_data = [
    {
        "question": "What is Python?",
        "reference": "Python is a programming language.",
        "answer": "Python is a programming language."
    },
    {
        "question": "What is RAG?",
        "reference": "RAG retrieves relevant information and uses it to generate an answer.",
        "answer": "RAG combines retrieval with language generation."
    },
    {
        "question": "What is a vector database?",
        "reference": "A vector database stores and searches vector embeddings.",
        "answer": "A vector database stores images only."
    }
]

eval_df = pd.DataFrame(eval_data)
eval_df

,question,reference,answer
0,What is Python?,Python is a programming language.,Python is a programming language.
1,What is RAG?,RAG retrieves relevant information and uses it...,RAG combines retrieval with language generation.
2,What is a vector database?,A vector database stores and searches vector e...,A vector database stores images only.


In [2]:
#Basic correctness checks

def keyword_score(answer, required_terms):
  answer = answer.lower()
  matched = sum(term.lower() in answer for term in required_terms)
  return matched / len(required_terms) if required_terms else 1.0

eval_df["score"] = [
    keyword_score(a, terms)
    for a, terms in zip(
        eval_df["answer"],
        [
            ["python", "programming language"],
            ["retrieval", "generation"],
            ["embeddings", "search"]
        ]
    )
]
eval_df[["question", "answer", "score"]]

,question,answer,score
0,What is Python?,Python is a programming language.,1.0
1,What is RAG?,RAG combines retrieval with language generation.,1.0
2,What is a vector database?,A vector database stores images only.,0.0


In [3]:
#Compare prompt versions

def mock_llm(prompt,question):
  if "example" in prompt.lower():
    return (
         "RAG retrieves relevant documents before generation. "
          "Example: a chatbot searches company policies before answering."
    )
  if "three bullet points" in prompt.lower():
    return (
            "- Retrieves relevant documents\n"
            "- Adds context to the prompt\n"
            "- Generates a context-informed answer"
        )
  return "RAG combines retrieval with language generation."

prompts = {
    "v1": "Explain RAG.",
    "v2": "Explain RAG in three bullet points for a beginner.",
    "v3": "Explain RAG in three bullet points and include a simple example."
}

question = "Explain RAG"

for version, prompt in prompts.items():
    answer = mock_llm(prompt, question)
    print(f"\n{version}\n{answer}")



v1
RAG combines retrieval with language generation.

v2
- Retrieves relevant documents
- Adds context to the prompt
- Generates a context-informed answer

v3
RAG retrieves relevant documents before generation. Example: a chatbot searches company policies before answering.


In [4]:
#Prompt evaluation using a rubric

def prompt_metrics(answer):
  return{
      "has_bullets": int("- " in answer),
      "has_example": int("example" in answer.lower()),
      "word_count": len(answer.split())
  }

results = []

for version,prompt in prompts.items():
  answer = mock_llm(prompt,question)
  results.append({
      "version": version,
      "answer": answer,
      **prompt_metrics(answer)
  })

pd.DataFrame(results)

,version,answer,has_bullets,has_example,word_count
0,v1,RAG combines retrieval with language generation.,0,0,6
1,v2,- Retrieves relevant documents\n- Adds context...,1,0,15
2,v3,RAG retrieves relevant documents before genera...,0,1,14


In [6]:
#Evaluate a sample RAG pipeline

documents = {
    "d1": "Python dictionaries store key-value pairs.",
    "d2": "A Python list is an ordered collection of items.",
    "d3": "RAG retrieves relevant documents and supplies them to an LLM.",
    "d4": "Vector databases store embeddings for similarity search."
}

def simple_retrieve(query, docs, k=2):
    query_words = set(query.lower().split())

    scores = {}
    for doc_id, text in docs.items():
        doc_words = set(text.lower().replace(".", "").split())
        scores[doc_id] = len(query_words & doc_words)

    return sorted(scores, key=scores.get, reverse=True)[:k]

query = "How does RAG retrieve documents for an LLM?"
retrieved_ids = simple_retrieve(query, documents)

print("Retrieved documents:")
for doc_id in retrieved_ids:
    print(doc_id, documents[doc_id])

Retrieved documents:
d3 RAG retrieves relevant documents and supplies them to an LLM.
d2 A Python list is an ordered collection of items.


In [8]:
#test whether the relevant document was retrieved

relevant_ids = {"d3"}

retrieved_set = set(retrieved_ids)

retrieval_recall = (
    len(retrieved_set & relevant_ids) / len(relevant_ids)
    if relevant_ids else 0
)

retrieval_precision = (
    len(retrieved_set & relevant_ids) / len(retrieved_set)
    if retrieved_set else 0
)

print("Retrieval precision:", retrieval_precision)
print("Retrieval recall:", retrieval_recall)

Retrieval precision: 0.5
Retrieval recall: 1.0


In [1]:
#Calculate retrieval metrics

import numpy as np

def precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    if not top_k:
        return 0.0
    return sum(d in relevant for d in top_k) / len(top_k)

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    return sum(d in relevant for d in retrieved[:k]) / len(relevant)

retrieved = ["D3", "D2", "D1"]
relevant = {"D1", "D3", "D5"}
graded_relevance = {"D1": 2, "D3": 3, "D5": 1}

print("Precision@3:", precision_at_k(retrieved, relevant, 3))
print("Recall@3:", recall_at_k(retrieved, relevant, 3))

Precision@3: 0.6666666666666666
Recall@3: 0.6666666666666666


In [2]:
#Track estimated token usage

def estimate_tokens(text):
    return max(1, round(len(text) / 4)) if text else 0

prompt = """
You are a helpful Python assistant.
Context: A dictionary stores key-value pairs.
Question: What is a dictionary?
"""

response = "A dictionary stores data as key-value pairs."

input_tokens = estimate_tokens(prompt)
output_tokens = estimate_tokens(response)

print("Estimated input tokens:", input_tokens)
print("Estimated output tokens:", output_tokens)
print("Estimated total tokens:", input_tokens + output_tokens)

Estimated input tokens: 29
Estimated output tokens: 11
Estimated total tokens: 40


In [3]:
#Build a cost calculator

def calculate_cost(
    input_tokens,
    output_tokens,
    input_price_per_million,
    output_price_per_million
):
    input_cost = input_tokens * input_price_per_million / 1_000_000
    output_cost = output_tokens * output_price_per_million / 1_000_000

    return input_cost + output_cost

input_tokens = 1200
output_tokens = 300

cost = calculate_cost(
    input_tokens=input_tokens,
    output_tokens=output_tokens,
    input_price_per_million=0.50,
    output_price_per_million=1.50
)

print(f"Estimated request cost: ${cost:.6f}")
print(f"Estimated cost for 10,000 requests: ${cost * 10_000:.2f}")

Estimated request cost: $0.001050
Estimated cost for 10,000 requests: $10.50


In [4]:
#Validate input

def validate_input(user_input):
    if not isinstance(user_input, str):
        return False, "Input must be text."

    if not user_input.strip():
        return False, "Input cannot be empty."

    if len(user_input) > 2000:
        return False, "Input exceeds the 2000-character limit."

    return True, "Input accepted."

tests = [
    "",
    "Explain RAG.",
    "x" * 2001
]

for item in tests:
    print(validate_input(item))

(False, 'Input cannot be empty.')
(True, 'Input accepted.')
(False, 'Input exceeds the 2000-character limit.')
